In [1]:
import pathlib
import sys

_root = pathlib.Path().resolve()
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
if str(_root.parent) not in sys.path and (_root.parent / 'src').exists():
    sys.path.insert(0, str(_root.parent))


# 02 Consumer Experience Intelligence

**Research question:** How are delivery performance and freight costs associated with customer review scores?
**Sources:** `analytics.fact_reviews` (grain `review_id,order_id`; preserves duplicate review IDs linked to different orders), `analytics.fact_delivery` (order grain). **Filter:** `valid_review_score IS NOT NULL`, delivered orders for delivery metrics/correlations as stated.

In [2]:
import pandas as pd

from src.db import get_connection

conn = get_connection(read_only=True)


In [3]:
# Grain: review-order joined to order delivery (chronological buckets)
by_bucket = conn.execute("""
SELECT fd.delivery_time_bucket,
       COUNT(*) AS reviews,
       AVG(fr.valid_review_score) AS avg_review
FROM analytics.fact_reviews fr
JOIN analytics.fact_delivery fd ON fr.order_id = fd.order_id
WHERE fr.valid_review_score IS NOT NULL
  AND fd.delivery_time_bucket IS NOT NULL
GROUP BY fd.delivery_time_bucket
""").fetchdf()
bucket_order = ['0-3 days','4-7 days','8-14 days','15-30 days','30+ days']
by_bucket['delivery_time_bucket'] = pd.Categorical(by_bucket['delivery_time_bucket'], categories=bucket_order, ordered=True)
by_bucket = by_bucket.sort_values('delivery_time_bucket').reset_index(drop=True)
by_bucket

,delivery_time_bucket,reviews,avg_review
0,0-3 days,6951,4.463530
1,4-7 days,23730,4.396924
2,8-14 days,37986,4.298057
3,15-30 days,23485,3.939578
4,30+ days,4207,2.204184


In [4]:
# On-time vs late (grain: review-order/delivery join). Late = delivery_gap_days < 0
timing = conn.execute("""
SELECT CASE WHEN fd.delivery_gap_days >= 0 THEN 'on_time_or_early' ELSE 'late' END AS timing,
       COUNT(*) AS reviews,
       AVG(fr.valid_review_score) AS avg_review
FROM analytics.fact_reviews fr
JOIN analytics.fact_delivery fd ON fr.order_id = fd.order_id
WHERE fr.valid_review_score IS NOT NULL AND fd.delivery_gap_days IS NOT NULL AND fd.is_delivered
GROUP BY 1
""").fetchdf()
timing

,timing,reviews,avg_review
0,late,6410,2.271139
1,on_time_or_early,89949,4.289842


In [5]:
# Pearson and Spearman correlations (explicit methods). Associations, not causation.
exp = conn.execute("""
SELECT fr.valid_review_score AS review_score,
       fd.delivery_time_days, fd.delivery_gap_days,
       fd.order_freight_value, fd.freight_share
FROM analytics.fact_reviews fr
JOIN analytics.fact_delivery fd ON fr.order_id = fd.order_id
WHERE fr.valid_review_score IS NOT NULL AND fd.is_delivered
""").fetchdf()
cols = ['review_score','delivery_time_days','delivery_gap_days','order_freight_value','freight_share']
corr = pd.DataFrame({
  'pearson': exp[cols].corr(method='pearson')['review_score'],
  'spearman': exp[cols].corr(method='spearman')['review_score'],
})
corr

,pearson,spearman
review_score,1.000000,1.000000
delivery_time_days,-0.333676,-0.234710
delivery_gap_days,0.266764,0.176114
order_freight_value,-0.089955,-0.088471
freight_share,-0.026362,-0.030552


In [6]:
conn.close()
